<a href="https://colab.research.google.com/github/LEIXINREN/TheLongDark_Save_Unpack/blob/main/the_long_dark_save_to_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base64 Encoding

In [26]:
import base64
modified_file = "sandbox1"
output_b64_file = "base64_save.txt"

with open(modified_file, "rb") as f:
    binary_data = f.read()

encoded_data = base64.b64encode(binary_data).decode("utf-8")

with open(output_b64_file, "w", encoding="utf-8") as f:
    f.write(encoded_data)

print(f"{output_b64_file}")

base64_save.txt


# LZF Decoding

In [27]:
!pip install python-lzf

In [28]:
import base64
import lzf

input_file = "base64_save.txt"
output_file = "lzf_base64_save.txt"

with open(input_file, "rb") as f:
    binary_content = f.read()

encoded_data = binary_content.decode("utf-8", errors="ignore").strip()
decoded_bytes = base64.b64decode(encoded_data)
print(f"Base64 decoding completed. Size: {len(decoded_bytes)} bytes")

# Automatically guess decompressed size by starting at 2x and scaling up
decompressed_bytes = None
original_size = len(decoded_bytes) * 2  # Initial guess size

for i in range(10):  # Maximum of 10 attempts
    try:
        # lzf.decompress requires: (data, expected_decompressed_size)
        decompressed_bytes = lzf.decompress(decoded_bytes, original_size)
        print(f"Decompression successful using buffer size: {original_size} bytes")
        break
    except Exception as e:
        # If output buffer is too small, double the size and try again
        original_size *= 2
        if i == 9:
            print(f"Decompression failed. Final error: {e}")

if decompressed_bytes:
    with open(output_file, "wb") as f:
        f.write(decompressed_bytes)
    print(f"Successfully saved to {output_file}. Final size: {len(decompressed_bytes)} bytes")
else:
    print("Could not complete decompression. Verify if the file is indeed LZF-compressed.")

Base64 decoding completed. Size: 11007215 bytes
Decompression successful using buffer size: 22014430 bytes
Successfully saved to lzf_base64_save.txt. Final size: 11433187 bytes


# Base64 Decoding


In [29]:
import base64
import json
from google.colab import files

def lzf_decompress(compressed_data: bytes) -> bytes:
    i = 0
    out = bytearray()
    data_len = len(compressed_data)

    while i < data_len:
        ctrl = compressed_data[i]
        i += 1

        if ctrl < 32:
            # Literal run
            length = ctrl + 1
            out.extend(compressed_data[i : i + length])
            i += length
        else:
            # Back reference
            length = ctrl >> 5
            ref_offset = ((ctrl & 0x1F) << 8) + 1

            if length == 7:
                length += compressed_data[i]
                i += 1

            ref_offset += compressed_data[i]
            i += 1
            length += 2

            ref_pos = len(out) - ref_offset
            for _ in range(length):
                out.append(out[ref_pos])
                ref_pos += 1

    return bytes(out)

input_file = "lzf_base64_save.txt"
print(f" Processing static input file: {input_file}")

try:
    # Read directly from local binary file
    with open(input_file, 'rb') as f:
        raw_data = f.read()

    try:
        file_content = raw_data.decode('utf-8').strip()
        # Try to parse as JSON first
        try:
            save_data = json.loads(file_content)
            if "m_Dict" in save_data and "global" in save_data["m_Dict"]:
                b64_string = save_data["m_Dict"]["global"]
                print(" Detected full save structure. Extracted m_Dict['global'].")
            else:
                b64_string = file_content
                print(" No m_Dict detected, treating the whole file as base64.")
        except json.JSONDecodeError:
            # If not JSON, assume it's just the base64 string
            b64_string = file_content
            print(" Detected plain text, treating as base64.")
    except UnicodeDecodeError:
        # File is binary (not text/JSON), treat original raw data as the compressed base64 / binary chunk directly
        print(" Binary data detected, treating raw data directly as compressed bytes.")
        compressed_bytes = raw_data
        b64_string = None

    if b64_string is not None:
        # --- Base64 Fix ---
        padding_needed = (4 - len(b64_string) % 4) % 4
        if padding_needed:
            print(f" Adding {padding_needed} padding character(s) ('=') to fix the base64 length.")
            b64_string += "=" * padding_needed

        print(" Decoding Base64...")
        compressed_bytes = base64.b64decode(b64_string)

    print(" Decompressing LZF...")
    json_bytes = lzf_decompress(compressed_bytes)

    final_json = json_bytes.decode('utf-8', errors='replace')

    output_filename = f"decoded_save.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(final_json)

    print(f" Success! Saved as {output_filename}")
    #files.download(output_filename)

except Exception as e:
    print(f" Error processing file: {e}")

 Processing static input file: lzf_base64_save.txt
 Detected full save structure. Extracted m_Dict['global'].
 Decoding Base64...
 Decompressing LZF...
 Success! Saved as decoded_save.json


# \\" to "

In [36]:
file_path = 'decoded_save.json'


with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()


new_content = content.replace(r'\"', '"')


with open(file_path, 'w', encoding='utf-8') as f:
    f.write(new_content)
